## Demo Download: Einzelner Monat (ausführbar)

Die folgenden Zellen zeigen die vollständige Pipeline exemplarisch für den **ersten verfügbaren Tag (01.01.2023)**:  
ZIP herunterladen → entpacken → filtern → als Parquet speichern.

Der gesamte Durchlauf für alle 36 Monate erfolgt über `src/process_ist_daten.py`.

In [ ]:
import os
import zipfile
import requests
import pandas as pd
from pathlib import Path

# --- Pfade ---
DEMO_DIR = Path("data/demo")
DEMO_DIR.mkdir(parents=True, exist_ok=True)

# --- 1. ZIP herunterladen ---
zip_url  = "https://archive.opentransportdata.swiss/istdaten/2023/ist-daten-2023-01.zip"
zip_path = DEMO_DIR / "ist-daten-2023-01.zip"

if not zip_path.exists():
    print("Lade ist-daten-2023-01.zip herunter...")
    r = requests.get(zip_url, stream=True)
    with open(zip_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)
    print(f"Download abgeschlossen: {zip_path.stat().st_size / 1e6:.0f} MB")
else:
    print("ZIP bereits vorhanden, überspringe Download.")

# --- 2. Erste CSV entpacken ---
extract_dir = DEMO_DIR / "ist-daten-2023-01"
extract_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zf:
    all_csvs = sorted([n for n in zf.namelist() if n.endswith(".csv")])
    first_csv = all_csvs[0]
    zf.extract(first_csv, extract_dir)
    print(f"Entpackt: {first_csv}  ({len(all_csvs)} CSVs im ZIP)")

csv_path = extract_dir / first_csv

# --- 3. Filtern: nur VBZ Tram ---
df = pd.read_csv(csv_path, sep=";", dtype=str, low_memory=False)
print(f"Gesamtzeilen: {len(df):,}")

mask = (df["BETREIBER_ID"] == "85:3849") & (df["PRODUKT_ID"] == "Tram")
df_vbz = df[mask]
print(f"VBZ Tram-Zeilen: {len(df_vbz):,}")

# --- 4. Als Parquet speichern ---
parquet_path = DEMO_DIR / (Path(first_csv).stem + ".parquet")
df_vbz.to_parquet(parquet_path, index=False, engine="pyarrow")

csv_size     = csv_path.stat().st_size / 1e6
parquet_size = parquet_path.stat().st_size / 1e6
print(f"\nCSV:     {csv_size:.0f} MB")
print(f"Parquet: {parquet_size:.1f} MB  (Faktor {csv_size/parquet_size:.0f}x kleiner)")
print(f"Gespeichert: {parquet_path}")

# --- 5. CSV aufräumen ---
csv_path.unlink()
print("CSV gelöscht.")